# A Small CLASS Fisher Forecast for SO-like CMB Constraints

This notebook installs CLASS, computes CMB power spectra with the `classy` Python wrapper, and builds a simple Fisher forecast for a Simons-Observatory-like small-aperture survey.

The main target is the tensor-to-scalar ratio `r`, but the code also forecasts the six usual LCDM parameters. We use a tiny nonzero fiducial value of `r` so CLASS can consistently compute tensor spectra and so the finite-difference derivative stays on the physical side `r > 0`. The calculation is intentionally simple and readable: it is meant as a teaching example, not a production SO likelihood.

References:

- CLASS: https://github.com/lesgourg/class_public
- Simons Observatory science goals and forecasts: https://arxiv.org/abs/1808.07445

A small CLASS-version detail: for tensors this notebook uses `modes = "s,t"` and `l_max_tensors`. Some examples on the web use slightly different tensor option names, but these are the names used by the current public CLASS explanatory file.


In [ ]:
# Install build tools and CLASS. This takes a few minutes in a fresh Colab runtime.
!apt-get -qq update
!apt-get -qq install -y build-essential gfortran git
!pip -q install cython numpy scipy matplotlib

%cd /content
![ -d class_public ] || git clone --depth 1 https://github.com/lesgourg/class_public.git
%cd /content/class_public
!make -j2
!pip -q install .

## Forecast Setup

We use an idealized SO SAT-like configuration for degree-scale B modes:

- sky fraction `f_sky = 0.10`
- polarization depth `Delta_P = 2 uK-arcmin`
- effective beam `theta_FWHM = 30 arcmin`
- multipole range `30 <= ell <= 300`

For simplicity, we also assign a temperature depth `Delta_T = Delta_P / sqrt(2)`. This is just a conventional white-noise relation used for a compact teaching forecast.

The Fisher matrix uses the Gaussian covariance of `TT/TE/EE`, plus an independent `BB` term. This ignores foregrounds, atmospheric filtering, calibration, delensing uncertainty, sky-mask mode coupling, and systematics, so the resulting `sigma(r)` is more optimistic than a realistic SO analysis.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

# Maximum multipole used in this simple SO SAT forecast.
LMAX = 300
LMIN = 30

# SO SAT-like survey assumptions-- see arXiv:2512.15833.
F_SKY = 0.10
DELTA_P_UK_ARCMIN = 2.0
DELTA_T_UK_ARCMIN = DELTA_P_UK_ARCMIN / math.sqrt(2.0)
THETA_FWHM_ARCMIN = 30.0

# We forecast six LCDM parameters plus r.
# ln10_10_A_s means ln(10^10 A_s), which is easier to step numerically than A_s itself.
PARAMETERS = [
    "omega_b",
    "omega_cdm",
    "h",
    "tau_reio",
    "ln10_10_A_s",
    "n_s",
    "r",
]

# A Planck-like flat LCDM fiducial cosmology.
# We use a tiny nonzero r so CLASS computes tensor spectra cleanly and
# the finite-difference derivative can stay at positive r values.
FIDUCIAL = {
    "omega_b": 0.02237,
    "omega_cdm": 0.1200,
    "h": 0.6736,
    "tau_reio": 0.0544,
    "ln10_10_A_s": 3.044,
    "n_s": 0.965,
    "r": 0.001,
}

# Finite-difference steps. These are deliberately not tiny, because Boltzmann-code
# numerical derivatives become noisy if the steps are too small.
STEPS = {
    "omega_b": 8.0e-5,
    "omega_cdm": 7.0e-4,
    "h": 0.003,
    "tau_reio": 0.004,
    "ln10_10_A_s": 0.008,
    "n_s": 0.003,
    "r": 0.0002,
}

## CLASS Helper Functions

CLASS wants `A_s`, while we forecast `ln(10^10 A_s)`. The helper below converts between them, runs CLASS, and returns lensed spectra in `uK^2`.

In [ ]:
def class_input_from_params(values):

    return {
        # Ask CLASS for temperature, polarization, and lensing spectra.
        "output": "tCl,pCl,lCl",
        "lensing": "yes",
        "l_max_scalars": LMAX,
        "l_max_tensors": LMAX,

        # Include scalar and tensor perturbations so BB depends on r.
        "modes": "s,t",
        "tensor_method": "exact",
        "r": values["r"],
        "k_pivot": 0.05,

        # LCDM parameters.
        "omega_b": values["omega_b"],
        "omega_cdm": values["omega_cdm"],
        "h": values["h"],
        "tau_reio": values["tau_reio"],
        "ln10^{10}A_s": values["ln10_10_A_s"],
        "n_s": values["n_s"],
        "T_cmb": 2.7255,
    }


def compute_lensed_cls(values, lmax=LMAX):
    """Run CLASS and return lensed TT, EE, BB, TE spectra in uK^2."""
    cosmo = Class()
    cosmo.set(class_input_from_params(values))
    cosmo.compute()

    raw_cls = cosmo.raw_cl(lmax)
    cls = cosmo.lensed_cl(lmax)
    tcmb_uK = cosmo.T_cmb() * 1.0e6

    spectra = {
        "ell": cls["ell"].copy(),
        "tt": cls["tt"].copy() * tcmb_uK**2,
        "ee": cls["ee"].copy() * tcmb_uK**2,
        "bb": cls["bb"].copy() * tcmb_uK**2,
        "tbb": raw_cls["bb"].copy() * tcmb_uK**2, #This just gives the GW contribution to the BB power spectrum
        "te": cls["te"].copy() * tcmb_uK**2,
    }

    cosmo.struct_cleanup()
    cosmo.empty()
    return spectra


fid_cls = compute_lensed_cls(FIDUCIAL)
print("Computed fiducial CLASS spectra up to ell =", fid_cls["ell"][-1])

## Plot the Fiducial BB Spectrum

This plot shows the signal that mostly drives the `r` constraint: lensing BB plus any tensor BB. Since the fiducial has only a tiny tensor amplitude, the plotted fiducial BB is almost entirely lensing BB.

In [ ]:
ell = fid_cls["ell"]
tbb_dell = ell * (ell + 1) * fid_cls["tbb"] / (2 * np.pi)
bb_dell = ell * (ell + 1) * fid_cls["bb"] / (2 * np.pi)

plt.figure(figsize=(7, 4.5))
plt.loglog(ell[2:], tbb_dell[2:], color="red", lw=2, label="tensor BB")
plt.loglog(ell[2:], bb_dell[2:], color="black", lw=2, label=f"lensed BB with r={FIDUCIAL['r']}")
plt.xlim(2, LMAX)
plt.xlabel(r"Multipole $\ell$")
plt.ylabel(r"$D_\ell^{BB}$ [$\mu K^2$]")
plt.title("Fiducial lensing BB spectrum from CLASS")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Instrument Noise

For white map noise and a Gaussian beam, the noise power spectrum is

\[
N_\ell = (\Delta\,\pi/180/60)^2
\exp\left[\frac{\ell(\ell+1)\theta_{\rm FWHM}^2}{8\ln 2}\right].
\]

Here `Delta` is in `uK-arcmin` and `theta_FWHM` is in radians.

In [ ]:
def cmb_noise_spectrum(delta_uK_arcmin, theta_fwhm_arcmin, lmax=LMAX):
    """Return a beam-deconvolved white-noise power spectrum in uK^2."""
    ell = np.arange(lmax + 1, dtype=float)
    arcmin_to_rad = np.pi / (180.0 * 60.0)
    delta = delta_uK_arcmin * arcmin_to_rad
    theta = theta_fwhm_arcmin * arcmin_to_rad
    return delta**2 * np.exp(ell * (ell + 1.0) * theta**2 / (8.0 * np.log(2.0)))


N_TT = cmb_noise_spectrum(DELTA_T_UK_ARCMIN, THETA_FWHM_ARCMIN)
N_EE = cmb_noise_spectrum(DELTA_P_UK_ARCMIN, THETA_FWHM_ARCMIN)
N_BB = N_EE.copy()

plt.figure(figsize=(7, 4.5))

ell_plot = ell[2:]
Cbb_total = fid_cls["bb"][2:] + N_BB[2:]

sigma_Cbb = np.sqrt(2.0 / ((2.0 * ell_plot + 1.0) * F_SKY)) * Cbb_total
sigma_Dbb = ell_plot * (ell_plot + 1.0) * sigma_Cbb / (2.0 * np.pi)

plt.plot(ell_plot, sigma_Dbb, label=r"$\sigma(D_\ell^{BB})$")
plt.loglog(ell[2:], tbb_dell[2:], color="red", lw=2, label=f"tensor BB with r={FIDUCIAL['r']}")
plt.plot(ell_plot, bb_dell[2:], label=f"lensed BB with r={FIDUCIAL['r']}")

plt.xlim(2, LMAX)
plt.yscale("log")
plt.xlabel(r"Multipole $\ell$")
plt.ylabel(r"$D_\ell$ [$\mu K^2$]")
plt.legend()
plt.show()

## Numerical Derivatives

The Fisher matrix needs derivatives like `d C_ell / d theta_i`. We estimate these with centered finite differences.

For `r`, we use a tiny positive fiducial value and a smaller finite-difference step, so the centered derivative never asks CLASS to evaluate an unphysical negative tensor amplitude. Since the tensor contribution to the CMB spectra is linear in `r` for these purposes, this gives the same derivative we would want near `r = 0`.

In [ ]:
def compute_derivatives(fiducial, steps):
    """Compute finite-difference derivatives of TT, TE, EE, BB."""
    derivs = {}

    for par in PARAMETERS:
        step = steps[par]

        low = fiducial.copy()
        high = fiducial.copy()

        if par == "r" and fiducial["r"] - step <= 0.0:
            # Safety fallback: if someone later chooses a step larger than r_fid,
            # use a one-sided derivative so CLASS never sees negative r.
            high[par] += step
            high_cls = compute_lensed_cls(high)
            derivs[par] = {
                spec: (high_cls[spec] - fid_cls[spec]) / step
                for spec in ["tt", "te", "ee", "bb"]
            }
        else:
            # Centered derivative. With the default r_fid=1e-4 and step=5e-5,
            # this is also used for r.
            low[par] -= step
            high[par] += step

            low_cls = compute_lensed_cls(low)
            high_cls = compute_lensed_cls(high)

            derivs[par] = {
                spec: (high_cls[spec] - low_cls[spec]) / (2.0 * step)
                for spec in ["tt", "te", "ee", "bb"]
            }

        print("finished derivative for", par)

    return derivs


derivs = compute_derivatives(FIDUCIAL, STEPS)

## Fisher Matrix

For temperature and E-mode polarization, we use the `2 x 2` covariance matrix

$$
\mathbf{C}_\ell =
\begin{pmatrix}
C_\ell^{TT}+N_\ell^{TT} & C_\ell^{TE} \\
C_\ell^{TE} & C_\ell^{EE}+N_\ell^{EE}
\end{pmatrix}.
$$

The Fisher contribution is

$$
F_{ij}^{T/E}
=
\sum_\ell
\frac{2\ell+1}{2}
f_{\rm sky}
\,
{\rm Tr}
\left[
\mathbf{C}_\ell^{-1}
\frac{\partial \mathbf{C}_\ell}{\partial \theta_i}
\mathbf{C}_\ell^{-1}
\frac{\partial \mathbf{C}_\ell}{\partial \theta_j}
\right].
$$

For BB we add

$$
F_{ij}^{BB}
=
\sum_\ell
\frac{2\ell+1}{2}
f_{\rm sky}
\frac{
\frac{\partial C_\ell^{BB}}{\partial \theta_i}
\frac{\partial C_\ell^{BB}}{\partial \theta_j}
}{
\left(C_\ell^{BB}+N_\ell^{BB}\right)^2
}.
$$

In [ ]:
def fisher_matrix(include_tt_te_ee=True, include_bb=True):
    """Build a Gaussian CMB Fisher matrix from the fiducial spectra and derivatives."""
    npar = len(PARAMETERS)
    fisher = np.zeros((npar, npar))

    for i, pi in enumerate(PARAMETERS):
        for j, pj in enumerate(PARAMETERS[i:], start=i):
            total = 0.0

            for l in range(LMIN, LMAX + 1):
                if include_tt_te_ee:
                    C = np.array([
                        [fid_cls["tt"][l] + N_TT[l], fid_cls["te"][l]],
                        [fid_cls["te"][l], fid_cls["ee"][l] + N_EE[l]],
                    ])
                    Cinv = np.linalg.inv(C)

                    dC_i = np.array([
                        [derivs[pi]["tt"][l], derivs[pi]["te"][l]],
                        [derivs[pi]["te"][l], derivs[pi]["ee"][l]],
                    ])
                    dC_j = np.array([
                        [derivs[pj]["tt"][l], derivs[pj]["te"][l]],
                        [derivs[pj]["te"][l], derivs[pj]["ee"][l]],
                    ])

                    total += (2*l + 1) * F_SKY / 2.0 * np.trace(Cinv @ dC_i @ Cinv @ dC_j)

                if include_bb:
                    Cbb = fid_cls["bb"][l] + N_BB[l]
                    total += (
                        (2*l + 1) * F_SKY / 2.0
                        * derivs[pi]["bb"][l]
                        * derivs[pj]["bb"][l]
                        / Cbb**2
                    )

            fisher[i, j] = total
            fisher[j, i] = total

    return fisher


F_all = fisher_matrix(include_tt_te_ee=True, include_bb=True)
F_bb_only = fisher_matrix(include_tt_te_ee=False, include_bb=True)

print("Fisher matrix shape:", F_all.shape)

## Convert Fisher Matrix to Forecast Errors

The covariance matrix is the inverse Fisher matrix. Because this teaching example is intentionally minimal and can have strong parameter degeneracies, we use a pseudo-inverse for numerical stability.

In [ ]:
def summarize_errors(F, title):
    """Print marginalized one-sigma errors from a Fisher matrix."""
    cov = np.linalg.pinv(F, rcond=1e-12)
    sigma = np.sqrt(np.diag(cov))

    print(title)
    print("-" * len(title))
    for par, fid, err in zip(PARAMETERS, [FIDUCIAL[p] for p in PARAMETERS], sigma):
        print(f"{par:14s} fiducial = {fid: .6g}   sigma = {err: .6g}")
    print()
    r_index = PARAMETERS.index("r")
    sigma_r_fixed_others = 1.0 / np.sqrt(F[r_index, r_index])
    print(f"marginalized sigma(r) = {sigma[r_index]:.6g}")
    # print(f"fixed-other-parameters sigma(r) = {sigma_r_fixed_others:.6g}")
    # print(f"For a true value very close to zero, 1.645 * fixed-parameter sigma(r) = {1.645 * sigma_r_fixed_others:.6g}")
    print()
    return sigma, cov


sigma_all, cov_all = summarize_errors(F_all, "TT/TE/EE/BB Fisher forecast")
sigma_bb, cov_bb = summarize_errors(F_bb_only, "BB-only Fisher forecast")

## A Useful Sanity Check

The BB-only marginalized LCDM errors are not very meaningful, because BB alone does not constrain all LCDM parameters well. The marginalized BB-only `sigma(r)` is much larger because BB alone cannot constrain all of LCDM. The full `TT/TE/EE/BB` case uses the same idealized SO Small Aperature Telescope (SAT) map for all spectra and is still not a realistic SO likelihood. In reality the Large Aperature Telescope is used to generate even more sensitive TT/TE/EE spectra.

```
# This is formatted as code
```



In [ ]:
labels = PARAMETERS
x = np.arange(len(labels))

plt.figure(figsize=(9, 4.5))
plt.semilogy(x - 0.15, sigma_all, "o", label="TT/TE/EE/BB")
plt.semilogy(x + 0.15, sigma_bb, "s", label="BB only")
plt.xticks(x, labels, rotation=35, ha="right")
plt.ylabel("Marginalized 1-sigma error")
plt.title("Simple SO-like Fisher forecast")
plt.grid(alpha=0.25, axis="y")
plt.legend()
plt.tight_layout()
plt.show()

## Final Compact Uncertainty Table

Run this cell after the Fisher cells above. It collects the fiducial values and marginalized errors in one place.


In [ ]:
# Final compact uncertainty table.
# This cell assumes the earlier cells have already computed sigma_all and sigma_bb.

def format_float(x):
    return f"{x:.4e}" if abs(x) < 1e-2 or abs(x) >= 1e3 else f"{x:.6g}"

header = f"{'parameter':14s} {'fiducial':>14s} {'sigma TT/TE/EE/BB':>20s} {'sigma BB-only':>16s}"
print(header)
print("-" * len(header))
for i, par in enumerate(PARAMETERS):
    print(
        f"{par:14s} "
        f"{format_float(FIDUCIAL[par]):>14s} "
        f"{format_float(sigma_all[i]):>20s} "
        f"{format_float(sigma_bb[i]):>16s}"
    )

r_index = PARAMETERS.index("r")
sigma_r_fixed_all = 1.0 / np.sqrt(F_all[r_index, r_index])
sigma_r_fixed_bb = 1.0 / np.sqrt(F_bb_only[r_index, r_index])
print()
print(f"Marginalized sigma(r) from TT/TE/EE/BB = {sigma_all[r_index]:.6g}")
print(f"Marginalized BB-only sigma(r) = {sigma_bb[r_index]:.6g}")
print(f"Fixed-LCDM r-only sigma(r) from TT/TE/EE/BB = {sigma_r_fixed_all:.6g}")
print(f"Fixed-LCDM r-only sigma(r) from BB only = {sigma_r_fixed_bb:.6g}")
print(f"If the true r is near zero, a one-sided 95% scale is 1.645 * fixed-LCDM BB-only sigma(r) = {1.645 * sigma_r_fixed_bb:.6g}.")


## Re-do this but with noise parameters corresponding to LiteBIRD

Look up the noise properties of LiteBIRD, a CMB satellite lead by the Japanese space agency JAXA. In the process of looking the numbers up, note the main mission goals of the project.